# PICKO Research · NB3 — **Separation**: can it tell look-alike tools apart?

Train one **40-tool model**, then probe curated groups of near-identical tools (same action across
sources, or same source across actions). Each group is small enough to offer in full at inference, so we
measure pure disambiguation + which tool it confuses for which.

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab: Runtime → Change runtime type → GPU (T4) first.** This cell clones the repo, pins the exact
JAX/Flax, mounts Drive (so checkpoints survive a restart), and sets the output dir. **Running locally?**
It's a no-op — just skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/picko"):
        !git clone -b hadar-work https://github.com/HadarBit/picko.git /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["PICKO_OUT_DIR"] = "/content/drive/MyDrive/picko_out"; os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    import shutil
    src, dst = "/content/drive/MyDrive/picko_balanced.jsonl", "/content/picko/data/picko_balanced.jsonl"
    if os.path.exists(src) and not os.path.exists(dst): shutil.copy(src, dst)
    print("GPU:")
    !nvidia-smi -L
    assert os.path.exists(dst), "Data missing: it ships in the repo clone; if absent, upload picko_balanced.jsonl to /content/picko/data/ or Drive root."
    print("bootstrap OK · OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally (CPU).")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()

### The 40 focus tools\nOne row per tool, with its family, category and **parameter count / bucket**.

In [ ]:
display(tools_dataframe(cat, FOCUS))

### All examples for these 40 tools\nOne row per training example (query → gold tool), tagged with the gold tool's **param bucket**.

In [ ]:
ex_df = examples_dataframe(cat, raw, FOCUS)
print("examples:", ex_df.shape[0], "| per param bucket:", ex_df["param_bucket"].value_counts().to_dict())
display(ex_df.head(10))

## 2 · The ambiguous groups

In [ ]:
grp_rows = []
for g, tools in SIMILAR_GROUPS.items():
    fams = sorted({family_of(t) for t in tools})
    buckets = sorted({param_bucket(cat.params_of(t)[1]) for t in tools})
    grp_rows.append({"group": g, "n_tools": len(tools), "families": ",".join(fams),
                     "param_buckets": ",".join(buckets), "tools": ", ".join(tools)})
groups_df = pd.DataFrame(grp_rows)
display(groups_df)

## 3 · Train (or reuse) the 40-tool model

In [ ]:
CAP_PER_TOOL, EPOCHS, EVAL_SUBSAMPLE, BATCH_SIZE = 40, 1, 40, 16
RUN_TRAIN, FORCE_RETRAIN = True, False
FOCUS40 = finetune_and_eval(cat, raw, tok, FOCUS, "focus40", OUT_DIR,
                            cap=CAP_PER_TOOL, epochs=EPOCHS, compact=False, token_aware=True,
                            eval_subsample=EVAL_SUBSAMPLE, run_train=RUN_TRAIN,
                            force_retrain=FORCE_RETRAIN, batch_size=BATCH_SIZE)
m40, p40, tk40 = FOCUS40["bundle"]
print("focus40 overall selection_acc:", FOCUS40["metrics"]["selection_acc"])

## 4 · Per-group disambiguation

In [ ]:
sep_rows, group_conf = [], {}
for gname, gtools in SIMILAR_GROUPS.items():
    gset = cat.restrict_dataset(raw, gtools, offer_all_max=len(gtools), cap_per_tool=CAP_PER_TOOL, seed=0)
    _, _, gtest = _per_tool_split(gset)
    if EVAL_SUBSAMPLE: gtest = gtest[:EVAL_SUBSAMPLE]
    gpreds = predict(m40, p40, tk40, gtest)
    gm = evaluate(gtest, gpreds, family_of=family_of)
    sep_rows.append({"group": gname, "n_tools": len(gtools), "n": gm["n"],
                     "selection_acc": gm["selection_acc"], "name_f1": gm["name_f1"]})
    group_conf[gname] = confusion(gtest, gpreds)
separation = pd.DataFrame(sep_rows).sort_values("selection_acc")
import json as _json
_json.dump({"per_group": sep_rows}, open(os.path.join(OUT_DIR, "separation_results.json"), "w"), indent=2)
display(separation)

plt.figure(figsize=(8,4)); plt.barh(separation["group"], separation["selection_acc"], color="#4C72B0")
plt.xlim(0,1); plt.xlabel("tool-selection accuracy"); plt.title("Separation: hardest look-alike groups (lower = more confused)")
plt.tight_layout(); plt.show()

## 5 · Confusion heatmaps (who gets mistaken for whom)

In [ ]:
for gname, conf in group_conf.items():
    labels = sorted(set(conf) | {p for row in conf.values() for p in row})
    M = pd.DataFrame(0, index=sorted(conf), columns=labels)
    for r, row in conf.items():
        for p, n in row.items(): M.loc[r, p] = n
    plt.figure(figsize=(0.9*len(labels)+2, 0.5*len(M)+1.5))
    if sns: sns.heatmap(M, annot=True, fmt="d", cmap="Blues", cbar=False)
    else:
        plt.imshow(M.values, cmap="Blues"); plt.xticks(range(len(labels)), labels, rotation=90); plt.yticks(range(len(M)), M.index)
    plt.title(f"Separation · {gname}"); plt.xlabel("predicted"); plt.ylabel("reference")
    plt.tight_layout(); plt.show()

## 6 · Read-out

Residual selection errors concentrate inside these look-alike groups. The lowest-accuracy group is the
frontier for a tool-picker; the heatmaps show whether confusions are symmetric (two tools mutually
confused) or a sink (everything collapses to one generic tool).